# Introduction to Keras

## 1. Classifying bank complaints

### 1.1 Loading the data

#### 1.1.1 Look at the data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from keras.utils.np_utils import to_categorical
from sklearn import preprocessing
from keras.preprocessing.text import Tokenizer

df = pd.read_csv('Bank_complaints.csv')
df.head()

In [ ]:
import random
random.seed(123)
df = df.sample(10000)

In [ ]:
df.index = range(10000)

In [ ]:
product = df["Product"]

There are 7 types of complaints

In [ ]:
product.value_counts()

In [ ]:
df.info()

In [ ]:
complaints = df["Consumer complaint narrative"]

### 1.1.2 One-hot encoding of the complaints

Only keep 2,000 most common words and use one-hot encoding

In [ ]:
tokenizer = Tokenizer(num_words=2000)

In [ ]:
tokenizer.fit_on_texts(complaints)

In [ ]:
sequences = tokenizer.texts_to_sequences(complaints)

In [ ]:
one_hot_results= tokenizer.texts_to_matrix(complaints, mode='binary')

word_index = tokenizer.word_index

print('Found %s unique tokens.' % len(word_index))

In [ ]:
#word_index.items()

In [ ]:
np.shape(one_hot_results)

Decode the review again, can be done as follows:

word_index stores the mapping from the label number to the actual word.

In [ ]:
reverse_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_review = ' '.join([reverse_index.get(i) for i in sequences[19]])

In [ ]:
decoded_review

In [ ]:
complaints[19]

You can see that some words dropped, but hopefully we caught the ones that have the useful distincive characteristics!

### 1.1.3 convert the products to numerical categories

In [ ]:
le = preprocessing.LabelEncoder()

In [ ]:
le.fit(product)

In [ ]:
list(le.classes_)

In [ ]:
product_cat = le.transform(product) 

In [ ]:
list(le.inverse_transform([0, 1, 3, 3, 0, 6, 4]))

In [ ]:
product_cat

In [ ]:
product_onehot = to_categorical(product_cat)

In [ ]:
np.shape(product_onehot)

### 1.1.4 Train - test split

In [ ]:
import random
random.seed(123)
test_index = random.sample(range(1,10000), 1500)

Create a test and a training set

In [ ]:
test = one_hot_results[test_index]
train = np.delete(one_hot_results, test_index, 0)

In [ ]:
label_test = product_onehot[test_index]
label_train = np.delete(product_onehot, test_index, 0)

In [ ]:
print(np.shape(label_test))
print(np.shape(label_train))
print(np.shape(test))
print(np.shape(train))

# 1.2 Building the network

Let's build a fully connected (Dense) layer network with relu activations in Keras. You can do this using: `Dense(16, activation='relu')`.

In this examples, use 2 hidden with 50 units in the first layer and 25 in the second, both with a `relu` activation function. Because we are dealing with a multiclass problem (classifying the complaints into 7 ), we use a use a softmax classifyer in order to output 7 class probabilities per case.

In [ ]:
from keras import models
from keras import layers
model = models.Sequential()
model.add(layers.Dense(50, activation='relu', input_shape=(2000,))) #2 hidden layers
model.add(layers.Dense(25, activation='relu'))
model.add(layers.Dense(7, activation='softmax'))

### 1.3 Compiling the model and look at the results

In [ ]:
model.compile(optimizer='SGD',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In the compiler, you'll be passing the optimizer (SGD = stochastic gradient descent), loss function, and metrics. Train the model for 120 epochs in mini-batches of 256 samples.

In [ ]:
history = model.fit(train,
                    label_train,
                    epochs=120,
                    batch_size=256)

The dictionary `history` two four entries:the loss and the accuracy achieved using the training set.

In [ ]:
history_dict = history.history
history_dict.keys()

### 1.5 Plot the results

In [ ]:
plt.clf()

import matplotlib.pyplot as plt
history_dict = history.history
loss_values = history_dict['loss']

epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, 'g', label='Training loss')

plt.title('Training loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plt.clf()

acc_values = history_dict['acc'] 

plt.plot(epochs, acc_values, 'r', label='Training acc')
plt.title('Training accuracy')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

It seems like we could just keep on going and accuracy would go up!

### 1.6 Make predictions

In [ ]:
model.predict(test)

In [ ]:
results_train = model.evaluate(train, label_train)

In [ ]:
results_test = model.evaluate(test, label_test)

In [ ]:
results_train

In [ ]:
results_test

We can see that the training set results are really good (a 89.4% classification accuracy!), but the test set results lag behind. In the next lab. We'll talk a little more about this in the next lecture, and will discuss how we can get better test set results as well!

# SOURCES

https://github.com/susanli2016/Machine-Learning-with-Python/blob/master/Consumer_complaints.ipynb

https://catalog.data.gov/dataset/consumer-complaint-database